# 02 · Partición train/test

Separamos los datos limpios en entrenamiento y test antes del análisis exploratorio. Así el EDA y todas las decisiones que salgan de él se toman solo con el train y no sufrimos fuga de información a los datos del test.

In [23]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

semilla = 42
ruta = "../data/silver/salida.csv"

df = pd.read_csv(ruta, parse_dates=["date_added"])

columnas_esperadas = ["location_id", "property_type", "price", "location", "city",
                      "latitude", "longitude", "baths", "bedrooms", "date_added",
                      "dist_centro_km", "Total_Area"]
faltan = set(columnas_esperadas) - set(df.columns)
assert not faltan, f"Faltan columnas en los datos limpios: {faltan}"

df.shape

(83630, 12)

## Análisis de anuncios iguales

La superficie y el precio son estándar en Pakistán, con parcelas de 5, 7 o 10 marla y precios redondos. Además los agentes publican la misma vivienda varias veces, simplemente cambiando algún detalle. Muchos anuncios coinciden por tanto en ciudad, zona, tipo, superficie y precio, y pueden o no ser la misma vivienda. Si uno cae en train y otro igual en test, el modelo habrá visto la respuesta antes de evaluarlo y provocamos la fuga de información. Por ello intentaremos evitar esta práctica.

In [24]:
grupos = ["city", "location", "property_type", "Total_Area", "price"]
grupo = df.groupby(grupos, sort=False).ngroup()

print(f"Anuncios: {len(df)} \nGrupos de posibles gemelos: {grupo.nunique()}")

Anuncios: 83630 
Grupos de posibles gemelos: 48547


## Justificación con datos de porqué no usar una partición aleatoria:

In [25]:
tr, te = train_test_split(df.index, test_size=0.2, random_state=semilla, stratify=df["city"])
gemelos = grupo.loc[te].isin(grupo.loc[tr])
print(f"Con una partición aleatoria los anuncios del test con un gemelo en train son: {gemelos.sum()} ({gemelos.mean():.1%})")

Con una partición aleatoria los anuncios del test con un gemelo en train son: 8835 (52.8%)


## Decisión 

Partición agrupada y estratificada con StratifiedGroupKFold

- Grupos: ciudad, zona, tipo, superficie y precio. Todos los posibles gemelos van juntos a train o a test.
- Estratificada por city: las ciudades están desequilibradas.


In [26]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=semilla)
idx_train, idx_test = next(sgkf.split(df, y=df["city"], groups=grupo))

train = df.iloc[idx_train].reset_index(drop=True)
test = df.iloc[idx_test].reset_index(drop=True)

print(f"Train: {len(train)} ({len(train) / len(df):.0%}) | Test: {len(test)} ({len(test) / len(df):.0%})")
print("Grupos compartidos:", len(set(grupo.iloc[idx_train]) & set(grupo.iloc[idx_test])))

Train: 66903 (80%) | Test: 16727 (20%)
Grupos compartidos: 0


In [27]:
comparacion = pd.DataFrame({
    "train": train["city"].value_counts(normalize=True),
    "test": test["city"].value_counts(normalize=True),
}).round(3)
display(comparacion)

for nombre, datos in [("train", train), ("test", test)]:
    print(f"Precio mediano {nombre}: {datos['price'].median():,.0f} €")

,train,test
city,,
Karachi,0.362,0.363
Lahore,0.342,0.342
Islamabad,0.151,0.151
Rawalpindi,0.106,0.106
Faisalabad,0.038,0.038


Precio mediano train: 76,833 €
Precio mediano test: 77,947 €


Train y test por tanto tienen la misma proporción de cada ciudad y la misma distribución del precio, y ninguna posible vivienda repetida está en los dos.

Guardamos los dos grupos de datos, train y test y en 03_EDA se trabajará solo con train.csv.

In [28]:
Path("../data/silver").mkdir(parents=True, exist_ok=True)
train.to_csv("../data/silver/train.csv", index=False)
test.to_csv("../data/silver/test.csv", index=False)
print("Guardados train.csv y test.csv en data/silver/")

Guardados train.csv y test.csv en data/silver/
